In [26]:
import pandas as pd
from scipy.stats import ttest_rel
import numpy as np

### Functions

In [27]:
def add_f1_score(results_file_path: str):
    excel_file = pd.ExcelFile(results_file_path)
    sheet_names = excel_file.sheet_names
    for sheet in sheet_names:
        updated_results = pd.read_excel(results_file_path, sheet_name=sheet)
        if "Experiment Reports" in sheet:
            # Calculate F1 scores
            f1_standard  = 2 * (updated_results["Standard Precision"] * updated_results["Standard Recall"]) / (updated_results["Standard Precision"] + updated_results["Standard Recall"])
            f1_invariant = 2 * (updated_results["Invariant Precision"] * updated_results["Invariant Recall"]) / (updated_results["Invariant Precision"] + updated_results["Invariant Recall"])
            f1_pcc = 2 * (updated_results["PCA Precision"] * updated_results["PCA Recall"]) / (updated_results["PCA Precision"] + updated_results["PCA Recall"])

            # Split DataFrame into before + after "PCA Recall"
            before = updated_results.loc[:, :"PCA Recall"]
            after = updated_results.loc[:, "Standard PCC":]

            # Insert new F1 columns in between
            updated_results = pd.concat([before, pd.DataFrame({"Standard F1 Score": f1_standard, "Invariant F1 Score": f1_invariant, "PCC F1 Score": f1_pcc}), after], axis=1)
            
            # Saving the result
            all_sheets = pd.read_excel(results_file_path, sheet_name=None)
            all_sheets[sheet] = updated_results

            with pd.ExcelWriter(results_file_path, engine="openpyxl") as writer:
                for sheet_name, df in all_sheets.items():
                    df.to_excel(writer, sheet_name=sheet_name, index=False)
        else:
            # Calculate F1-Score
            updated_results["F1 Score"] = (2 * updated_results["Precision"] * updated_results["Recall"]) / (updated_results["Precision"] + updated_results["Recall"])
            
            # Saving the result
            all_sheets = pd.read_excel(results_file_path, sheet_name=None)
            all_sheets[sheet] = updated_results

            with pd.ExcelWriter(results_file_path, engine="openpyxl") as writer:
                for sheet_name, df in all_sheets.items():
                    df.to_excel(writer, sheet_name=sheet_name, index=False)

def calculate_test_averages(input_path: str):
    excel_file = pd.ExcelFile(input_path)
    sheet_names = excel_file.sheet_names

    average_results = None
    for sheet in sheet_names:
        if "Experiment Reports" in sheet:
            df = pd.read_excel(input_path, sheet_name=sheet)

            # Get Model Name
            model_name = sheet.replace("Experiment Reports (", "")[:-1]

            # Group experiments
            num = df["Test Name"].str.extract(r"Test[_\s]*(\d+)", expand=False)
            df["Test Group"] = num

            # Get average metrics
            metrics = ["Standard Accuracy", "Invariant Accuracy", "PCA Accuracy",
                       "Standard F1 Score", "Invariant F1 Score", "PCC F1 Score",
                       "Standard PCC", "Invariant PCC", "PCA PCC"]
            averages_df = df.groupby("Test Group")[metrics].mean().reset_index()
            averages_df = averages_df.round(4)
            
            # Update Test Group name
            averages_df["Test Group"] = "Test " + averages_df["Test Group"] + " (" + model_name + ")"

            if average_results is None:
                average_results = averages_df
            else:
                average_results = pd.concat([average_results, averages_df], ignore_index=True)

    return average_results

def paired_pval(group, cond_a, cond_b, metric):
    a = group[group["Test Name"].str.contains(cond_a, case=False, regex=False)].set_index("Fold Number")
    b = group[group["Test Name"].str.contains(cond_b, case=False, regex=False)].set_index("Fold Number")
    if a.empty or b.empty:
        return np.nan
    common = a.index.intersection(b.index)
    if len(common) < 2:  # need at least 2 pairs
        return np.nan
    a = a.loc[common, metric]
    b = b.loc[common, metric]
    _, p = ttest_rel(a, b)
    return float(np.round(p, 4))

def run_for_df(df: pd.DataFrame) -> pd.DataFrame:
    # Extract the base test name (drop trailing condition tag)
    df["Test Group"] = df["Test Name"].str.replace(r"_standard|_invariant|_pca", "", regex=True)

    rows = []
    for test, group in df.groupby("Test Group"):
        # Std vs Inv
        p_acc_std_inv = paired_pval(group, "standard", "invariant", "Accuracy")
        p_f1_std_inv  = paired_pval(group, "standard", "invariant", "F1 Score")

        # Inv vs PCA
        p_acc_inv_pca = paired_pval(group, "invariant", "pca", "Accuracy")
        p_f1_inv_pca  = paired_pval(group, "invariant", "pca", "F1 Score")

        rows.append({
            "Test": test,
            "Accuracy P-Value (Std vs Inv)": p_acc_std_inv,
            "Accuracy Significant (Std vs Inv)": (p_acc_std_inv < 0.05) if pd.notna(p_acc_std_inv) else np.nan,
            "F1 Score P-Value (Std vs Inv)": p_f1_std_inv,
            "F1 Score Significant (Std vs Inv)": (p_f1_std_inv < 0.05) if pd.notna(p_f1_std_inv) else np.nan,
            "Accuracy P-Value (Inv vs PCA)": p_acc_inv_pca,
            "Accuracy Significant (Inv vs PCA)": (p_acc_inv_pca < 0.05) if pd.notna(p_acc_inv_pca) else np.nan,
            "F1 Score P-Value (Inv vs PCA)": p_f1_inv_pca,
            "F1 Score Significant (Inv vs PCA)": (p_f1_inv_pca < 0.05) if pd.notna(p_f1_inv_pca) else np.nan,
        })

    return pd.DataFrame(rows)

def statistical_significance_all(file_path: str, output_file: str):
    # Map model tag -> per-fold sheet name
    model_sheets = {
        "LR":   "Per Fold Results (LR)",
        "NN":   "Per Fold Results (NN)",
        "LSTM": "Per Fold Results (LSTM)",
    }

    # Run and write all in one workbook
    with pd.ExcelWriter(f"Results/{output_file}", engine="openpyxl") as writer:
        for model_tag, sheet in model_sheets.items():
            try:
                df = pd.read_excel(file_path, sheet_name=sheet)
            except ValueError:
                # Sheet not found; skip
                continue

            results_df = run_for_df(df)
            # (Optional) consistent rounding of numeric columns
            num_cols = results_df.select_dtypes(include=["float", "int"]).columns
            results_df[num_cols] = results_df[num_cols].round(4)

            results_df.to_excel(writer, sheet_name=f"{model_tag} Paired T-Tests", index=False)

def count_significant_accuracy(file_path: str, sheet_name: str) -> dict:
    df = pd.read_excel(file_path, sheet_name=sheet_name)

    percentages = {}
    for col in df.columns:
        if "Accuracy Significant" in col:
            total = df[col].notna().sum()  # how many comparisons exist
            if total > 0:
                percentages[col] = str(round((df[col].sum() / total) * 100, 2))
            else:
                percentages[col] = None  # no comparisons available
    
    for percentage in percentages:
        print(f" {percentage} : {percentages[percentage]}%")

def count_significant_f1(file_path: str, sheet_name: str) -> dict:
    df = pd.read_excel(file_path, sheet_name=sheet_name)

    percentages = {}
    for col in df.columns:
        if "F1 Score Significant" in col:
            total = df[col].notna().sum()  # how many comparisons exist
            if total > 0:
                percentages[col] = str(round((df[col].sum() / total) * 100, 2))
            else:
                percentages[col] = None  # no comparisons available
    
    for percentage in percentages:
        if percentages[percentage] == None:
            print(f" {percentage} : {percentages[percentage]}")
        else:
            print(f" {percentage} : {percentages[percentage]}%")

### Running Functions

In [28]:
# Adding F1 Score
add_f1_score(results_file_path="Results/AGAIN_Results.xlsx")
add_f1_score(results_file_path="Results/RECOLA_Results.xlsx")

# Calculate Test Averages
again_average = calculate_test_averages(input_path="Results/AGAIN_Results.xlsx")
recola_average = calculate_test_averages(input_path="Results/RECOLA_Results.xlsx")

with pd.ExcelWriter("Results/Average_Results.xlsx", engine="openpyxl") as writer:
    again_average.to_excel(writer, sheet_name="AGAIN", index=False)
    recola_average.to_excel(writer, sheet_name="RECOLA", index=False)

# Statistical Significance Tests
statistical_significance_all(file_path = "Results/AGAIN_Results.xlsx", output_file="Statistical_Significance_AGAIN.xlsx")
statistical_significance_all(file_path = "Results/RECOLA_Results.xlsx", output_file="Statistical_Significance_RECOLA.xlsx")

In [29]:
print("AGAIN Significance (Accuracy)")
file_path = "Results/Statistical_Significance_AGAIN.xlsx"
print("Logistic Regression:"); count_significant_accuracy(file_path, "LR Paired T-Tests")
print("Neural Network:"); count_significant_accuracy(file_path, "NN Paired T-Tests")
print("LSTM:"); count_significant_accuracy(file_path, "LSTM Paired T-Tests")

print("\nAGAIN Significance (F1 Score)")
file_path = "Results/Statistical_Significance_AGAIN.xlsx"
print("Logistic Regression:"); count_significant_f1(file_path, "LR Paired T-Tests")
print("Neural Network:"); count_significant_f1(file_path, "NN Paired T-Tests")
print("LSTM:"); count_significant_f1(file_path, "LSTM Paired T-Tests")

AGAIN Significance (Accuracy)
Logistic Regression:
 Accuracy Significant (Std vs Inv) : 0.0%
 Accuracy Significant (Inv vs PCA) : 0.0%
Neural Network:
 Accuracy Significant (Std vs Inv) : 3.33%
 Accuracy Significant (Inv vs PCA) : 6.67%
LSTM:
 Accuracy Significant (Std vs Inv) : 0.0%
 Accuracy Significant (Inv vs PCA) : 13.33%

AGAIN Significance (F1 Score)
Logistic Regression:
 F1 Score Significant (Std vs Inv) : 29.17%
 F1 Score Significant (Inv vs PCA) : 25.0%
Neural Network:
 F1 Score Significant (Std vs Inv) : 25.0%
 F1 Score Significant (Inv vs PCA) : 29.17%
LSTM:
 F1 Score Significant (Std vs Inv) : 0.0%
 F1 Score Significant (Inv vs PCA) : 16.67%


In [30]:
print("\nRECOLA Significance (Accuracy)")
file_path = "Results/Statistical_Significance_RECOLA.xlsx"
print("Logistic Regression:"); count_significant_accuracy(file_path, "LR Paired T-Tests")
print("Neural Network:"); count_significant_accuracy(file_path, "NN Paired T-Tests")
print("LSTM:"); count_significant_accuracy(file_path, "LSTM Paired T-Tests")

print("\nRECOLA Significance (F1 Score)")
file_path = "Results/Statistical_Significance_RECOLA.xlsx"
print("Logistic Regression:"); count_significant_f1(file_path, "LR Paired T-Tests")
print("Neural Network:"); count_significant_f1(file_path, "NN Paired T-Tests")
print("LSTM:"); count_significant_f1(file_path, "LSTM Paired T-Tests")


RECOLA Significance (Accuracy)
Logistic Regression:
 Accuracy Significant (Std vs Inv) : 11.11%
 Accuracy Significant (Inv vs PCA) : 16.67%
Neural Network:
 Accuracy Significant (Std vs Inv) : 13.89%
 Accuracy Significant (Inv vs PCA) : 15.28%
LSTM:
 Accuracy Significant (Std vs Inv) : 0.0%
 Accuracy Significant (Inv vs PCA) : 0.0%

RECOLA Significance (F1 Score)
Logistic Regression:
 F1 Score Significant (Std vs Inv) : 1.47%
 F1 Score Significant (Inv vs PCA) : 0.0%
Neural Network:
 F1 Score Significant (Std vs Inv) : 3.39%
 F1 Score Significant (Inv vs PCA) : 0.0%
LSTM:
 F1 Score Significant (Std vs Inv) : None
 F1 Score Significant (Inv vs PCA) : None
